In [2]:
pip install pandas duckdb deltalake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.9 MB/s eta 0:00:00


In [3]:
import json
import os
import duckdb
import pandas as pd
from deltalake import DeltaTable, write_deltalake

# Criar diretórios do Lakehouse
os.makedirs("/content/lakehouse/bronze", exist_ok=True)
os.makedirs("/content/lakehouse/silver", exist_ok=True)
os.makedirs("/content/lakehouse/gold", exist_ok=True)

# Simulação de lote de dados brutos (Raw/Landing)
raw_data = [
    {
        "order_id": 101,
        "customer_id": "C1",
        "category": "Eletronicos",
        "amount": 1200.50,
        "ts": "2026-08-30 10:00:00",
    },
    {
        "order_id": 102,
        "customer_id": "C2",
        "category": "Moveis",
        "amount": 450.00,
        "ts": "2026-08-30 10:05:00",
    },
    {
        "order_id": 103,
        "customer_id": None,
        "category": "Livros",
        "amount": 80.00,
        "ts": "2026-08-30 10:10:00",
    },  # Dado inconsistente (sem customer)
    {
        "order_id": 101,
        "customer_id": "C1",
        "category": "Eletronicos",
        "amount": 1200.50,
        "ts": "2026-08-30 10:00:00",
    },  # Duplicata
    {
        "order_id": 104,
        "customer_id": "C3",
        "category": "Eletronicos",
        "amount": 2300.00,
        "ts": "2026-08-30 10:15:00",
    },
]

with open("/content/raw_events.json", "w") as f:
  json.dump(raw_data, f)

print("✅ Ambiente e dados brutos gerados com sucesso!")

✅ Ambiente e dados brutos gerados com sucesso!


In [4]:
# Ingestão para DataFrame
df_raw = pd.read_json("/content/raw_events.json")

# Gravando na Bronze com Delta Lake
bronze_path = "/content/lakehouse/bronze/orders"
write_deltalake(bronze_path, df_raw, mode="append")

# Consultando via DuckDB
con = duckdb.connect()
print("--- Camada Bronze (Tabela Delta) ---")
con.execute(f"SELECT * FROM delta_scan('{bronze_path}')").df()

--- Camada Bronze (Tabela Delta) ---


,order_id,customer_id,category,amount,ts
0,101,C1,Eletronicos,1200.5,2026-08-30 10:00:00
1,102,C2,Moveis,450.0,2026-08-30 10:05:00
2,103,None,Livros,80.0,2026-08-30 10:10:00
3,101,C1,Eletronicos,1200.5,2026-08-30 10:00:00
4,104,C3,Eletronicos,2300.0,2026-08-30 10:15:00


In [5]:
# Leitura da Bronze
df_bronze = con.execute(f"SELECT * FROM delta_scan('{bronze_path}')").df()

# Limpeza e Deduplicação
df_silver = df_bronze.dropna(subset=["customer_id"]).drop_duplicates(
    subset=["order_id"]
)

# Gravando na Silver
silver_path = "/content/lakehouse/silver/orders"
write_deltalake(silver_path, df_silver, mode="overwrite")

print("--- Camada Silver (Dados Tratados) ---")
con.execute(f"SELECT * FROM delta_scan('{silver_path}')").df()

--- Camada Silver (Dados Tratados) ---


,order_id,customer_id,category,amount,ts,__index_level_0__
0,101,C1,Eletronicos,1200.5,2026-08-30 10:00:00,0
1,102,C2,Moveis,450.0,2026-08-30 10:05:00,1
2,104,C3,Eletronicos,2300.0,2026-08-30 10:15:00,4


In [6]:
# Agregação SQL usando DuckDB
gold_path = "/content/lakehouse/gold/sales_by_category"

df_gold = con.execute(f"""
    SELECT
        category,
        COUNT(order_id) AS total_pedidos,
        ROUND(SUM(amount), 2) AS receita_total,
        ROUND(AVG(amount), 2) AS ticket_medio
    FROM delta_scan('{silver_path}')
    GROUP BY category
""").df()

# Gravando na Gold
write_deltalake(gold_path, df_gold, mode="overwrite")

print("--- Camada Gold (Visão Executiva / BI) ---")
con.execute(f"SELECT * FROM delta_scan('{gold_path}')").df()

--- Camada Gold (Visão Executiva / BI) ---


,category,total_pedidos,receita_total,ticket_medio
0,Eletronicos,2,3500.5,1750.25
1,Moveis,1,450.0,450.00


In [7]:
# 1. Simular uma alteração acidental / atualização na camada Silver
df_update = df_silver.copy()
df_update["amount"] = (
    df_update["amount"] * 2
)  # Dobrando valores indevidamente
write_deltalake(silver_path, df_update, mode="overwrite")

# 2. Inspecionar o histórico da tabela Delta
dt = DeltaTable(silver_path)
history = dt.history()
print(
    f"Versões registradas na tabela: {len(history)} transações no log Delta."
)

# 3. Time Travel: Consultando a versão original (Versão 0)
df_original = con.execute(
    f"SELECT * FROM delta_scan('{silver_path}')"
).df()  # Versão atual (alterada)
dt.load_as_version(0)  # Carrega o estado da Versão 0

print("\n--- Dados da Versão 0 (Antes do update indevido) ---")
display(dt.to_pandas())

Versões registradas na tabela: 2 transações no log Delta.

--- Dados da Versão 0 (Antes do update indevido) ---


,order_id,customer_id,category,amount,ts,__index_level_0__
0,101,C1,Eletronicos,1200.5,2026-08-30 10:00:00,0
1,102,C2,Moveis,450.0,2026-08-30 10:05:00,1
2,104,C3,Eletronicos,2300.0,2026-08-30 10:15:00,4
